# Follower Formation Response Ablation Experiment

This notebook adds a follower-side experiment to the current multi-UAV dissertation work.

It directly addresses the question:

> If the leader UAV is dynamically reassigned, how do the follower UAVs maintain formation stability?

The experiment compares two variants:

1. **Leader-only reassignment**: the leader is reassigned using the perception-aware score, but followers directly track the new leader reference.  
2. **Leader reassignment + follower smoothing**: the same leader reassignment is used, but followers apply smoothed reference tracking and velocity synchronization after leader changes.

The purpose is to show that the contribution is not only leader selection, but also formation-preserving follower response after leader switches.


In [ ]:
# =============================
# 1. Import Libraries
# =============================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.patches import Circle

np.random.seed(42)

output_dir = Path("follower_response_results")
figure_dir = output_dir / "figures"

output_dir.mkdir(exist_ok=True)
figure_dir.mkdir(exist_ok=True)

print("Output directory:", output_dir)


# 2. Model Definition

Leader reassignment still uses the perception-aware leadership score:

$$
S_i = w_pP_i + w_cC_i + w_gG_i
$$

Follower target position is defined as:

$$
p_i^{ref}(t) = p_L(t) + r_i
$$

where $p_L(t)$ is the current leader position and $r_i$ is the formation offset.

For the follower smoothing method, the effective reference is:

$$
\tilde{p}_i^{ref}(t) = (1-\lambda)\tilde{p}_i^{ref}(t-1) + \lambda p_i^{ref}(t)
$$

This prevents sudden follower target jumps after leader reassignment.


In [ ]:
# =============================
# 2. Global Simulation Settings
# =============================

n_uavs = 5
steps = 120
world_size = 22

start = np.array([2.0, 2.0])
goal = np.array([19.0, 19.0])

formation_offsets = np.array([
    [0.0, 0.0],
    [-1.2, -0.8],
    [-1.2, 0.8],
    [-2.4, -0.8],
    [-2.4, 0.8],
])

scenario_obstacles = {
    "dense": np.array([
        [5.5, 5.5, 1.3],
        [7.5, 7.0, 1.4],
        [10.0, 9.5, 1.5],
        [12.5, 12.0, 1.4],
        [15.0, 14.0, 1.3],
        [8.0, 13.5, 1.2],
        [13.5, 7.5, 1.2],
        [16.0, 9.5, 1.1],
    ]),
    "corridor": np.array([
        [6.0, 4.5, 1.4],
        [6.0, 8.0, 1.4],
        [6.0, 11.5, 1.4],
        [11.0, 7.0, 1.4],
        [11.0, 10.5, 1.4],
        [15.5, 12.0, 1.4],
        [15.5, 15.5, 1.4],
        [10.0, 15.5, 1.2],
    ])
}

follower_modes = ["leader_only", "formation_smoothing"]

mode_titles = {
    "leader_only": "Leader-only Reassignment",
    "formation_smoothing": "Leader Reassignment + Follower Smoothing"
}

print("Settings loaded.")


In [ ]:
# =============================
# 3. Helper Functions
# =============================

def obstacle_repulsion(pos, obstacles):
    rep = np.zeros(2)
    min_clearance = 10.0

    for ox, oy, r in obstacles:
        obs = np.array([ox, oy])
        diff = pos - obs
        dist = np.linalg.norm(diff) + 1e-6
        clearance = dist - r
        min_clearance = min(min_clearance, clearance)

        if dist < r + 2.8:
            rep += (diff / dist) * (1.0 / max(clearance, 0.15)) * 0.55

    return rep, min_clearance


def perception_score(pos, obstacles):
    _, clearance = obstacle_repulsion(pos, obstacles)
    clearance_score = np.clip(clearance / 4.0, 0, 1)
    center_penalty = 0.12 * np.exp(-np.linalg.norm(pos - np.array([11, 11]))**2 / 35)
    return float(np.clip(clearance_score - center_penalty + np.random.normal(0, 0.025), 0, 1))


def communication_score(pos, centroid):
    dist = np.linalg.norm(pos - centroid)
    return float(np.clip(1 - dist / 6.5 + np.random.normal(0, 0.02), 0, 1))


def geometry_score(pos, centroid):
    dist = np.linalg.norm(pos - centroid)
    return float(np.clip(1 - dist / 5.5, 0, 1))


def pairwise_distances(positions):
    dists = []
    for i in range(len(positions)):
        for j in range(i + 1, len(positions)):
            dists.append(np.linalg.norm(positions[i] - positions[j]))
    return np.array(dists)

print("Helper functions loaded.")


In [ ]:
# =============================
# 4. Single-Trial Simulation
# =============================

def simulate_follower_response(
    scenario="dense",
    follower_mode="formation_smoothing",
    wp=0.62,
    wc=0.23,
    wg=0.15,
    delta=0.08,
    lambda_smooth=0.28,
    recovery_threshold=0.45
):
    """
    Simulate perception-aware leader reassignment with two follower response modes.

    follower_mode:
        leader_only: direct follower tracking after leader switch.
        formation_smoothing: smoothed follower reference tracking and velocity synchronization.
    """

    obstacles = scenario_obstacles[scenario]

    positions = np.array([start + offset for offset in formation_offsets], dtype=float)
    velocities = np.zeros_like(positions)
    smoothed_refs = positions.copy()

    trajectories = [[p.copy()] for p in positions]
    leader_history = []

    current_leader = 0
    switch_count = 0
    switch_times = []

    formation_errors = []
    collision_flags = []
    distance_violation_flags = []

    for t in range(steps):
        centroid = positions.mean(axis=0)

        # Leader scoring
        scores = []
        for p in positions:
            P = perception_score(p, obstacles)
            C = communication_score(p, centroid)
            G = geometry_score(p, centroid)
            scores.append(wp * P + wc * C + wg * G)

        candidate = int(np.argmax(scores))

        if scores[candidate] - scores[current_leader] > delta:
            new_leader = candidate
        else:
            new_leader = current_leader

        if new_leader != current_leader:
            switch_count += 1
            switch_times.append(t)

        current_leader = new_leader
        leader_history.append(current_leader)

        # Leader motion
        leader_pos = positions[current_leader]
        goal_vec = goal - leader_pos
        goal_vec = goal_vec / (np.linalg.norm(goal_vec) + 1e-6)
        rep, _ = obstacle_repulsion(leader_pos, obstacles)
        leader_velocity = 0.20 * goal_vec + 0.16 * rep
        leader_new = leader_pos + leader_velocity

        # Desired formation references around the current leader
        desired_refs = np.zeros_like(positions)
        for i in range(n_uavs):
            desired_refs[i] = leader_new + (formation_offsets[i] - formation_offsets[current_leader])

        new_positions = positions.copy()
        new_velocities = velocities.copy()

        for i in range(n_uavs):
            rep_i, _ = obstacle_repulsion(positions[i], obstacles)

            if i == current_leader:
                new_velocities[i] = leader_velocity
                new_positions[i] = positions[i] + new_velocities[i]
                smoothed_refs[i] = new_positions[i]
            else:
                if follower_mode == "leader_only":
                    reference = desired_refs[i]
                    kp = 0.22
                    kv = 0.00
                elif follower_mode == "formation_smoothing":
                    smoothed_refs[i] = (1 - lambda_smooth) * smoothed_refs[i] + lambda_smooth * desired_refs[i]
                    reference = smoothed_refs[i]
                    kp = 0.19
                    kv = 0.12
                else:
                    raise ValueError("Unknown follower_mode")

                tracking_velocity = kp * (reference - positions[i])
                sync_velocity = kv * (leader_velocity - velocities[i])
                new_velocities[i] = tracking_velocity + sync_velocity + 0.08 * rep_i
                new_positions[i] = positions[i] + new_velocities[i]

        positions = np.clip(new_positions, 0.5, world_size - 0.5)
        velocities = new_velocities

        for i in range(n_uavs):
            trajectories[i].append(positions[i].copy())

        # Step metrics
        rel = positions - positions[current_leader]
        target_rel = formation_offsets - formation_offsets[current_leader]
        formation_error = np.mean(np.linalg.norm(rel - target_rel, axis=1))
        formation_errors.append(formation_error)

        min_clearance = min(obstacle_repulsion(p, obstacles)[1] for p in positions)
        collision_flags.append(1 if min_clearance < 0.35 else 0)

        dists = pairwise_distances(positions)
        violation = np.any(dists < 0.55) or np.any(dists > 4.2)
        distance_violation_flags.append(1 if violation else 0)

    trajectories = [np.array(traj) for traj in trajectories]

    # Recovery time after leader switches
    recovery_records = []
    for st in switch_times:
        recovered = False
        for future_t in range(st, len(formation_errors)):
            if formation_errors[future_t] < recovery_threshold:
                recovery_records.append(future_t - st)
                recovered = True
                break
        if not recovered:
            recovery_records.append(steps - st)

    mean_recovery_time = 0.0 if len(recovery_records) == 0 else float(np.mean(recovery_records))

    final_positions = np.array([traj[-1] for traj in trajectories])
    final_dist = np.linalg.norm(final_positions.mean(axis=0) - goal)

    collision_risk = float(np.mean(collision_flags))
    formation_error_mean = float(np.mean(formation_errors))
    distance_violation_rate = float(np.mean(distance_violation_flags))
    navigation_success = float(np.clip(1 - final_dist / 22 - collision_risk, 0, 1))

    robustness_score = float(
        navigation_success
        - 0.25 * collision_risk
        - 0.15 * formation_error_mean
        - 0.10 * distance_violation_rate
        - 0.03 * switch_count / steps
    )

    return {
        "scenario": scenario,
        "follower_mode": follower_mode,
        "switch_count": switch_count,
        "mean_recovery_time": mean_recovery_time,
        "collision_risk": collision_risk,
        "formation_error": formation_error_mean,
        "distance_violation_rate": distance_violation_rate,
        "navigation_success": navigation_success,
        "robustness_score": robustness_score,
        "trajectories": trajectories,
        "leader_history": leader_history,
        "formation_errors": formation_errors,
        "switch_times": switch_times
    }

print("Follower-response simulation loaded.")


# 5. Run Monte Carlo Experiments

The follower ablation is evaluated in dense and corridor-like environments because these are the cases where leader switching and formation disturbance are most likely to occur.


In [ ]:
# =============================
# 5. Run Monte Carlo Experiments
# =============================

n_trials = 200
records = []

for scenario in ["dense", "corridor"]:
    for mode in follower_modes:
        for trial in range(n_trials):
            result = simulate_follower_response(scenario=scenario, follower_mode=mode)
            records.append({
                "scenario": scenario,
                "follower_mode": mode,
                "trial": trial + 1,
                "switch_count": result["switch_count"],
                "mean_recovery_time": result["mean_recovery_time"],
                "collision_risk": result["collision_risk"],
                "formation_error": result["formation_error"],
                "distance_violation_rate": result["distance_violation_rate"],
                "navigation_success": result["navigation_success"],
                "robustness_score": result["robustness_score"]
            })

follower_df = pd.DataFrame(records)

csv_path = output_dir / "follower_response_ablation_results.csv"
follower_df.to_csv(csv_path, index=False)

print("Saved:", csv_path)
follower_df.head()


In [ ]:
# =============================
# 6. Statistical Summary
# =============================

metrics = [
    "formation_error",
    "mean_recovery_time",
    "distance_violation_rate",
    "collision_risk",
    "navigation_success",
    "robustness_score",
    "switch_count"
]

summary_rows = []

for scenario in ["dense", "corridor"]:
    for mode in follower_modes:
        subset = follower_df[(follower_df["scenario"] == scenario) & (follower_df["follower_mode"] == mode)]
        row = {"scenario": scenario, "follower_mode": mode}

        for metric in metrics:
            mean = subset[metric].mean()
            std = subset[metric].std()
            ci95 = 1.96 * std / np.sqrt(len(subset))
            row[f"{metric}_mean"] = mean
            row[f"{metric}_std"] = std
            row[f"{metric}_ci95"] = ci95

        summary_rows.append(row)

follower_summary = pd.DataFrame(summary_rows)

summary_path = output_dir / "follower_response_ablation_summary.csv"
follower_summary.to_csv(summary_path, index=False)

print("Saved:", summary_path)
follower_summary


# 7. Main Result Visualization

These plots are designed to show whether follower smoothing reduces formation disturbance after leader switches.


In [ ]:
# =============================
# 7. Bar Plots with 95% CI
# =============================

for metric in metrics:
    plt.figure(figsize=(8, 5))

    labels = []
    means = []
    errors = []

    for scenario in ["dense", "corridor"]:
        for mode in follower_modes:
            row = follower_summary[(follower_summary["scenario"] == scenario) & (follower_summary["follower_mode"] == mode)].iloc[0]
            labels.append(f"{scenario}\n{mode}")
            means.append(row[f"{metric}_mean"])
            errors.append(row[f"{metric}_ci95"])

    plt.bar(range(len(means)), means, yerr=errors, capsize=4)
    plt.xticks(range(len(labels)), labels, rotation=25, ha="right")
    plt.ylabel(metric)
    plt.title(f"Follower Response Ablation: {metric}")
    plt.tight_layout()

    fig_path = figure_dir / f"follower_ablation_{metric}.png"
    plt.savefig(fig_path, dpi=300)
    plt.show()

    print("Saved:", fig_path)


# 8. Qualitative Trajectory Visualization

This section visualizes one representative dense scenario for both follower response modes.


In [ ]:
# =============================
# 8. Representative Trajectory Comparison
# =============================

example_results = {
    mode: simulate_follower_response(scenario="dense", follower_mode=mode)
    for mode in follower_modes
}

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, mode in zip(axes, follower_modes):
    result = example_results[mode]
    obstacles = scenario_obstacles["dense"]

    ax.set_title(mode_titles[mode])
    ax.set_xlim(0, world_size)
    ax.set_ylim(0, world_size)
    ax.set_aspect("equal")

    for ox, oy, r in obstacles:
        ax.add_patch(Circle((ox, oy), r, alpha=0.25))
        ax.add_patch(Circle((ox, oy), r + 0.6, alpha=0.08))

    ax.scatter(start[0], start[1], marker="s", s=80, label="Start")
    ax.scatter(goal[0], goal[1], marker="*", s=160, label="Goal")

    for i, traj in enumerate(result["trajectories"]):
        ax.plot(traj[:, 0], traj[:, 1], linewidth=1.4, alpha=0.85)
        ax.scatter(traj[-1, 0], traj[-1, 1], s=35)

    leader_pts = []
    for t, leader_id in enumerate(result["leader_history"]):
        leader_pts.append(result["trajectories"][leader_id][t])

    leader_pts = np.array(leader_pts)
    ax.plot(leader_pts[:, 0], leader_pts[:, 1], linestyle="--", linewidth=2.5, label="Leader path")

    ax.text(
        0.6,
        20.3,
        f"Switches: {result['switch_count']}\n"
        f"Form err: {result['formation_error']:.2f}\n"
        f"Recovery: {result['mean_recovery_time']:.2f}\n"
        f"Violation: {result['distance_violation_rate']:.2f}",
        fontsize=9,
        bbox=dict(boxstyle="round", alpha=0.15)
    )

    ax.grid(True, alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3)

plt.suptitle("Follower Response Ablation under Dense Environment", fontsize=14)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])

fig_path = figure_dir / "follower_response_trajectory_comparison.png"
plt.savefig(fig_path, dpi=300)
plt.show()

print("Saved:", fig_path)


In [ ]:
# =============================
# 9. Formation Error over Time
# =============================

plt.figure(figsize=(8, 5))

for mode in follower_modes:
    result = example_results[mode]
    plt.plot(result["formation_errors"], label=mode_titles[mode])
    for st in result["switch_times"]:
        plt.axvline(st, linestyle="--", alpha=0.15)

plt.xlabel("Time step")
plt.ylabel("Formation error")
plt.title("Formation Error Response after Leader Switches")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

fig_path = figure_dir / "formation_error_time_response.png"
plt.savefig(fig_path, dpi=300)
plt.show()

print("Saved:", fig_path)


# 10. Interpretation Guide

The expected dissertation interpretation is:

1. **Leader-only reassignment** can produce abrupt follower target changes after each leader switch. This may increase formation error, recovery time, and inter-UAV distance violation.

2. **Follower smoothing** reduces sudden reference jumps. It should reduce formation error and formation recovery time after leader changes.

3. The technical contribution can therefore be stated as:

> The proposed framework does not only reassign the leader based on perception quality. It also introduces a leader-switching-aware follower response mechanism that preserves formation stability after leader changes.

This experiment directly responds to the follower-side contribution issue.
